# Smart Logistics Tracker Japan — Kaizen Logistics IoT Data Simulation

This notebook generates a clean and realistic simulated IoT logistics dataset for **Kaizen Logistics**, a mock premier logistics company in Japan. The dataset is designed for blockchain-powered logistics tracking, time-series visualization, dashboard development, and sensor-based shipment monitoring.

## 1. Simulation Objective

The goal of this simulation is to create 100 package tracking records with realistic Japan-based locations, delivery statuses, temperature readings, and RFID tracking indicators. The generated data supports the project story of a smart logistics company that uses IoT sensors and blockchain records to improve package tracking, supply chain transparency, and fraud prevention.

The dataset is intentionally structured for later visualizations such as maps, temperature monitoring charts, RFID success indicators, delivery performance dashboards, and time-series line plots.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Configure reproducibility
rng = np.random.default_rng(148)

# Output filename
base_filename = "smart_logistics_tracker_japan_kaizenlogistics"

## 2. Load Japan City and Prefecture Reference Data

In [ ]:
# Locate the Japan cities reference file.
# When running from the repository root, place the file in the same folder as this notebook
# or inside the IOT Data Simulation folder.
candidate_paths = [
    Path("japan_prefectures_cities_729.csv"),
    Path("IOT Data Simulation") / "japan_prefectures_cities_729.csv",
    Path("../japan_prefectures_cities_729.csv")
]

location_path = None
for path in candidate_paths:
    if path.exists():
        location_path = path
        break

if location_path is None:
    raise FileNotFoundError(
        "japan_prefectures_cities_729.csv was not found. "
        "Place it beside this notebook or inside the IOT Data Simulation folder."
    )

locations = pd.read_csv(location_path)

# Split LatLng into numeric latitude and longitude
locations[["Latitude", "Longitude"]] = locations["LatLng"].astype(str).str.split(";", expand=True)
locations["Latitude"] = pd.to_numeric(locations["Latitude"].str.strip(), errors="coerce")
locations["Longitude"] = pd.to_numeric(locations["Longitude"].str.strip(), errors="coerce")
locations = locations.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)

print("Location reference loaded successfully.")
print("Total available city records:", len(locations))
locations.head()

## 3. Define Simulation Rules

In [ ]:
def estimate_ambient_temperature(latitude, status, perishable, rng):
    """
    Estimate a realistic Japan-based package temperature.
    The temperature is influenced by latitude, package type, and delivery status.
    """
    # Warmer values in southern Japan, cooler values in northern Japan
    ambient_temp = 28 - ((latitude - 24) * 0.52) + rng.normal(0, 1.8)

    if perishable == "YES":
        if status == "Delivered":
            # Successful cold-chain handling for most delivered perishables
            temp = rng.normal(5.2, 1.2)
        elif status == "In Transit":
            # Slightly wider cold-chain variation while still in movement
            temp = rng.normal(7.0, 2.2)
        else:
            # Not delivered packages may show stronger temperature excursions
            temp = rng.normal(14.5, 4.0)
    else:
        if status == "Delivered":
            temp = ambient_temp + rng.normal(0, 1.0)
        elif status == "In Transit":
            temp = ambient_temp + rng.normal(1.5, 2.0)
        else:
            temp = ambient_temp + rng.normal(3.0, 3.0)

    return round(float(np.clip(temp, -5, 32)), 1)


def classify_temperature_issue(temperature, perishable):
    """
    Classify temperature condition for dashboard-friendly reporting.
    """
    if perishable == "YES":
        if 0 <= temperature <= 8:
            return "Cool"
        if 8 < temperature <= 15:
            return "Ambient"
        return "Danger Zone"

    if temperature <= 10:
        return "Cool"
    if temperature <= 26:
        return "Ambient"
    return "Danger Zone"


def rfid_metrics(status, rng):
    """
    Generate RFID verification and success/failure values aligned with delivery outcome.
    """
    if status == "Delivered":
        failure = round(float(rng.uniform(0.1, 3.0)), 2)
        verified = rng.random() < 0.99
    elif status == "In Transit":
        failure = round(float(rng.uniform(2.0, 7.0)), 2)
        verified = rng.random() < 0.96
    else:
        failure = round(float(rng.uniform(16.0, 35.0)), 2)
        verified = rng.random() < 0.35

    success = round(100 - failure, 2)

    if failure <= 5:
        failure_label = "Low Risk"
    elif failure <= 15:
        failure_label = "Moderate Risk"
    else:
        failure_label = "High Risk"

    if success >= 97:
        success_label = "Excellent"
    elif success >= 90:
        success_label = "Good"
    else:
        success_label = "At Risk"

    return verified, failure, failure_label, success, success_label

## 4. Generate 100 Kaizen Logistics Package Records

In [ ]:
n_records = 100
package_ids = [f"PKG{i:03d}" for i in range(1, n_records + 1)]

# 96% delivered rate, with a small number of active or failed shipments for dashboard storytelling
statuses = (["Delivered"] * 96) + (["In Transit"] * 2) + (["Not Delivered"] * 2)
rng.shuffle(statuses)

start_base = pd.Timestamp("2026-05-01 08:00:00")
records = []

for idx, package_id in enumerate(package_ids):
    status = statuses[idx]

    origin = locations.sample(n=1, random_state=int(rng.integers(1, 1_000_000))).iloc[0]
    delivery = locations.sample(n=1, random_state=int(rng.integers(1, 1_000_000))).iloc[0]

    attempts = 0
    while delivery["City"] == origin["City"] and attempts < 10:
        delivery = locations.sample(n=1, random_state=int(rng.integers(1, 1_000_000))).iloc[0]
        attempts += 1

    order_date = start_base + pd.Timedelta(
        days=int(rng.integers(0, 22)),
        hours=int(rng.integers(0, 12)),
        minutes=int(rng.integers(0, 60)),
        seconds=int(rng.integers(0, 60)),
        microseconds=int(rng.integers(0, 999999))
    )

    if status == "Delivered":
        transit_hours = int(rng.integers(18, 96))
        delivery_date = order_date + pd.Timedelta(hours=transit_hours, minutes=int(rng.integers(0, 60)))
        timestamp = delivery_date

        current_city = delivery["City"]
        current_prefecture = delivery["Prefecture"]
        current_lat = delivery["Latitude"]
        current_lon = delivery["Longitude"]
        current_location = f"{current_city}, {current_prefecture}"

    elif status == "In Transit":
        timestamp = order_date + pd.Timedelta(hours=int(rng.integers(8, 72)), minutes=int(rng.integers(0, 60)))
        delivery_date = pd.NaT

        current = locations.sample(n=1, random_state=int(rng.integers(1, 1_000_000))).iloc[0]
        current_city = current["City"]
        current_prefecture = current["Prefecture"]
        current_lat = current["Latitude"]
        current_lon = current["Longitude"]
        current_location = f"{current_city} Transfer Center, {current_prefecture}"

    else:
        timestamp = order_date + pd.Timedelta(hours=int(rng.integers(24, 120)), minutes=int(rng.integers(0, 60)))
        delivery_date = pd.NaT

        current_city = delivery["City"]
        current_prefecture = delivery["Prefecture"]
        current_lat = delivery["Latitude"] + float(rng.normal(0, 0.03))
        current_lon = delivery["Longitude"] + float(rng.normal(0, 0.03))
        current_location = f"{current_city} Exception Hub, {current_prefecture}"

    perishable = "YES" if rng.random() < 0.34 else "NO"
    temperature = estimate_ambient_temperature(current_lat, status, perishable, rng)
    temp_issue = classify_temperature_issue(temperature, perishable)

    rfid_verified, rfid_failure, rfid_failure_label, rfid_success, rfid_success_label = rfid_metrics(status, rng)

    tracking_number = f"KZJP2026{idx + 1:06d}"
    rfid_number = f"RFID-KZ-{idx + 1:04d}"

    records.append({
        "package_id": package_id,
        "tracking_number": tracking_number,
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S.%f"),
        "Origin Location": f"{origin['City']}, {origin['Prefecture']}",
        "Origin City": origin["City"],
        "Origin Prefecture": origin["Prefecture"],
        "Order Date": order_date.strftime("%Y-%m-%d %H:%M:%S.%f"),
        "Current Location": current_location,
        "Delivery Date": "" if pd.isna(delivery_date) else delivery_date.strftime("%Y-%m-%d %H:%M:%S.%f"),
        "Status": status,
        "Perishable": perishable,
        "Temperature": temperature,
        "Temperature Issue": temp_issue,
        "Current Longitude": round(float(current_lon), 6),
        "Current Latitude": round(float(current_lat), 6),
        "Delivery Longitude": round(float(delivery["Longitude"]), 6),
        "Delivery Latitude": round(float(delivery["Latitude"]), 6),
        "Delivery City": delivery["City"],
        "Delivery Prefecture": delivery["Prefecture"],
        "RFID #": rfid_number,
        "RFID Verified": "YES" if rfid_verified else "NO",
        "RFID Failure %": rfid_failure,
        "RFID Failure Label": rfid_failure_label,
        "RFID Success %": rfid_success,
        "RFID Success Label": rfid_success_label
    })

kaizen_df = pd.DataFrame(records)

# Sort by timestamp for natural time-series use
kaizen_df["timestamp_dt"] = pd.to_datetime(kaizen_df["timestamp"])
kaizen_df = kaizen_df.sort_values("timestamp_dt").drop(columns=["timestamp_dt"]).reset_index(drop=True)

print("Generated records:", len(kaizen_df))
kaizen_df.head()

## 5. Validate Dataset Quality

In [ ]:
print("Dataset shape:", kaizen_df.shape)

print("\nDelivery status distribution:")
print(kaizen_df["Status"].value_counts())

print("\nDelivery rate:")
delivery_rate = (kaizen_df["Status"].eq("Delivered").mean()) * 100
print(f"{delivery_rate:.2f}%")

print("\nRFID success summary:")
print(kaizen_df["RFID Success %"].describe())

print("\nTemperature issue distribution:")
print(kaizen_df["Temperature Issue"].value_counts())

print("\nMissing values per column:")
print(kaizen_df.isna().sum())

## 6. Export CSV and JSON Files

In [ ]:
# Save outputs in a repository-friendly location.
# If the notebook is run from the repository root, outputs are saved to IOT Data Simulation.
# If the notebook is already inside IOT Data Simulation, outputs are saved to the current folder.
if Path.cwd().name == "IOT Data Simulation":
    output_dir = Path(".")
else:
    output_dir = Path("IOT Data Simulation")
    output_dir.mkdir(parents=True, exist_ok=True)

csv_output_path = output_dir / f"{base_filename}.csv"
json_output_path = output_dir / f"{base_filename}.json"

kaizen_df.to_csv(csv_output_path, index=False)

with json_output_path.open("w", encoding="utf-8") as f:
    json.dump(kaizen_df.to_dict(orient="records"), f, indent=2, ensure_ascii=False)

print(f"CSV file saved to: {csv_output_path}")
print(f"JSON file saved to: {json_output_path}")

## 7. Dataset Summary

The generated dataset represents 100 simulated package records for Kaizen Logistics in Japan. The delivery performance is intentionally high to reflect a strong logistics operation, while a small number of in-transit and not-delivered records remain for dashboard analysis and exception monitoring.

The dataset is suitable for:
- package tracking dashboards,
- geolocation-based route and delivery maps,
- temperature monitoring visualizations,
- RFID verification and reliability analysis,
- blockchain data storage and retrieval demonstrations.